# Caso 3. Deslizamientos de ladera inducidos por ciclones: cómo define ITCHI el peligro en el municipio y con cuánta anticipación fue alto

**Eventos.**

| Ciclón | Deslizamiento | Municipio | Hora local | UTC (usada) | Fuente |
|---|---|---|---|---|---|
| Manuel 2013 (EP132013) | Alud de La Pintada, Sierra de Atoyac | Atoyac de Álvarez, Guerrero | 16 sep 2013, ~15:30 | 16 sep 20:30 (± 1 h) | La Jornada (3 nov 2013), Proceso (21 sep 2013) |
| Earl 2016 (AL052016) | Deslizamientos en colonias de Huauchinango y en Xaltepec | Huauchinango, Puebla | 7 ago 2016, madrugada | 7 ago 08:00 (± 3 h) | Boletín ERN (8 ago 2016) |

En 2013 y 2016 el centro de México tenía horario de verano (UTC−5). Algunas fuentes secundarias fechan La Pintada el 14 de septiembre; los reportes de campo (pizarrón de la escuela, sobrevivientes) dan el lunes 16 a las 15:30. Se usa el 16.

**Dos hechos que condicionan la lectura:**
1. **No hay posición del ciclón al ocurrir el deslizamiento.** El centro de Manuel se disipó sobre México hacia las 06 UTC del 16 y se regeneró en el golfo de California el 17 (NHC, informe EP132013); el deslizamiento ocurrió ~14 h después de la última posición. Earl también se había disipado. ITCHI se define en las posiciones del ciclón, así que la serie de ITCHI se interrumpe antes del evento.
2. **Ingrid estaba activa** en el golfo de México al mismo tiempo que Manuel. La lluvia de Guerrero el 16 no es atribuible solo a Manuel; el informe del NHC lo dice explícitamente.

Por eso se muestran dos series:
- **ITCHI municipal** en las posiciones del ciclón (máximo sobre los píxeles del municipio), para las cuatro versiones: v0.2.0, P95, Saffir-Simpson y P95 + Saffir-Simpson.
- **H_P continuo**: el peligro por lluvia en todos los tiempos de 6 h, sin atribución al ciclón (no necesita su posición). Es lo que ITCHI daría si el ciclón siguiera definido.

**Cómo se mide la anticipación** (fijado antes de ver las series; `casos_v021.lead_times`). Para cada clase k y cada serie, con los tiempos ≤ t_evento en las 120 h previas:
- **inicio del episodio:** el comienzo del último tramo continuo (pasos de 6 h seguidos) con clase ≥ k;
- **anticipación** = t_evento − inicio del episodio;
- **horas desde el fin** = t_evento − último tiempo del episodio. Menos de 6 h significa que el peligro seguía alto al ocurrir el deslizamiento; más, que había bajado o que la serie se interrumpió (sin posición del ciclón).

Es anticipación **observada**: cuánto tiempo antes del deslizamiento la lluvia medida (MSWEP) ya ponía el municipio en clase ≥ k. Es una cota superior de lo que un pronóstico podría dar; la anticipación con pronósticos (AIFS) es la fase F5 del plan v0.3.

**Qué no hace ITCHI.** No sabe de pendiente, suelo ni uso del suelo. Un deslizamiento depende de la lluvia acumulada sobre una ladera susceptible; ITCHI solo puede decir que la lluvia fue climatológicamente extrema. Que ITCHI esté alto es necesario, no suficiente.

**Entradas.** `python casos_v021.py --casos MANUEL2013 EARL2016` (o todos).

## 0. Configuración y datos

In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from itchi.config import get_project_root

ROOT = get_project_root()
sys.path.insert(0, str(ROOT / "scripts"))
import casos_v021 as cv       # noqa: E402
import figuras_v021 as F      # noqa: E402

CFG = cv.ConfigV021()          # rutas (D:/MSWEP_V316_6H, climatologías, IBTrACS, ROCLOUD, municipios)
FORZAR = False                 # True: recalcula aunque exista el caché de outputs/diagnostics/avance_v021/
MODO_MUNICIPIO = "poligonos"   # "poligonos" (shapefile) o "malla" (cada píxel con el valor de su municipio)
FIG = CFG.fig_dir / "caso3_deslizamientos"
FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 30)


def guardar(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", dpi=200, bbox_inches="tight")
    plt.show()


assert CFG.mswep_6h_dir.is_dir(), f"no existe {CFG.mswep_6h_dir}"
for d, f in CFG.clim.items():
    assert f.exists(), f"falta {f.name}"

In [ ]:
EXP = ["v020", "p95", "ss", "p95ss"]
DECL = cv.load_declaratorias(CFG)
EVENTOS = {}
for c, d in cv.DESLIZAMIENTOS.items():
    C = cv.load_or_compute(c, CFG, force=FORZAR)
    code = cv.find_municipio(C["gdf"], d["municipio"], d["cve_ent"])
    t_ev = cv.t_event_utc(c)
    t_decl = cv.declaratoria_inicio(DECL, cv.CASES[c]["atcf"]).get(code, None)
    EVENTOS[c] = dict(C=C, code=code, t_event=t_ev, t_decl=t_decl, **d)
    tr = pd.DatetimeIndex(C["track"]["time"])
    print(f"{c}: {d['municipio']} ({code}) · evento {t_ev:%Y-%m-%d %H:%M} UTC · "
          f"última posición antes del evento {tr[tr <= t_ev].max():%Y-%m-%d %H} UTC · "
          f"declaratoria: {'—' if t_decl is None else f'{t_decl:%Y-%m-%d}'}")

## 1. Manuel 2013 y La Pintada (Atoyac de Álvarez)

### 1.1 Series del municipio

Cuatro paneles con el mismo eje de tiempo (UTC). La línea negra vertical es el deslizamiento; la roja discontinua, el inicio de la declaratoria (00 h locales); el sombreado gris, los tiempos sin posición del ciclón en IBTrACS.

**Cómo leerla.**
- **a) Lluvia en el píxel representativo** (el del municipio con mayor P6/P95 en el evento) y sus percentiles locales de cada duración. Muestra si el deslizamiento llegó con un pico de 6 h o con acumulación: en una ladera, R24 y A72 por encima de su P95 son más relevantes que un P6 aislado.
- **b) Percentil más alto superado** por P6, R24 y A72 en ese píxel. Es la «medida de los diferentes percentiles» en una sola línea por variable.
- **c) Peligro municipal.** ITCHI en las posiciones (puntos) para las cuatro versiones, H_W con Saffir-Simpson (triángulos) y H_P continuo (líneas finas) para los máximos P99 y P95.
- **d) Clase municipal.** Escalones de ITCHI por versión y del H_P continuo. La anticipación se lee como la distancia horizontal entre el inicio del último escalón ≥ k y la línea negra.

In [ ]:
E = EVENTOS["MANUEL2013"]
print(E["nota"])
guardar(F.fig_serie_deslizamiento(E["C"], E["code"], f"{E['municipio']} ({E['evento']})", E["t_event"], EXP,
                                  t_decl=E["t_decl"], window_h=(120, 48)), "manuel2013_atoyac_series")

### 1.2 Anticipación por nivel

Una fila por serie y clase. `anticipacion_h` es la respuesta a «con cuánta antelación el peligro tenía un valor alto»; `horas_desde_fin` dice si ese peligro seguía activo al ocurrir el deslizamiento.

**Cómo leerla.**
- Para ITCHI, `horas_desde_fin` ≥ 14 h es esperable: la última posición de Manuel es de las 06 UTC del 16. Lo que importa es en qué clase quedó el municipio en esa última posición y desde cuándo.
- Para el H_P continuo, `horas_desde_fin` < 6 h indica que la lluvia seguía siendo extrema al ocurrir el deslizamiento, ya sin ciclón definido.
- Comparar versiones: P95 solo puede adelantar el inicio (o no cambiarlo); Saffir-Simpson solo puede bajar la clase de viento de una tormenta tropical como Manuel (60 kt al tocar tierra).

In [ ]:
TAB_M = F.tabla_anticipacion(E["C"], E["code"], E["t_event"], EXP, levels=(2, 3, 4, 5), window_h=120)
TAB_M.to_csv(FIG / "manuel2013_atoyac_anticipacion.csv", index=False)
TAB_M.pivot_table(index="serie", columns="nivel", values="anticipacion_h", sort=False).round(1)

In [ ]:
TAB_M

### 1.3 Mapas antes del deslizamiento

Clase de ITCHI (v0.2.0 y propuesta P95 + Saffir-Simpson) en las posiciones 24, 12 y 0 h antes de la última posición previa al deslizamiento, en malla y por municipio; Atoyac de Álvarez con borde negro.

**Cómo leerla.** Muestra si el peligro en Atoyac era local (el municipio en el núcleo) o si era parte de una franja costera más amplia. Con la propuesta Saffir-Simpson, lo que quede en clase alta es lluvia, porque el viento de Manuel no pasa de la clase 2.

In [ ]:
def tiempos_previos(E, horas=(24, 12, 0)):
    times = pd.DatetimeIndex(E["C"]["ds"]["time"].values)
    ult = times[times <= E["t_event"]].max()
    return [t for t in (ult - pd.Timedelta(hours=h) for h in horas) if t in times]


for tag in ("v020", "p95ss"):
    guardar(F.fig_mapa_deslizamiento(E["C"], E["code"], tiempos_previos(E), tag=tag, mode=MODO_MUNICIPIO),
            f"manuel2013_atoyac_mapas_{tag}")

## 2. Earl 2016 y Huauchinango

Mismas figuras y tablas. La hora del deslizamiento es incierta (madrugada; se usa 03:00 local ± 3 h), así que las anticipaciones tienen ±3 h de incertidumbre.

In [ ]:
E = EVENTOS["EARL2016"]
print(E["nota"])
guardar(F.fig_serie_deslizamiento(E["C"], E["code"], f"{E['municipio']}", E["t_event"], EXP,
                                  t_decl=E["t_decl"], window_h=(120, 48)), "earl2016_huauchinango_series")
TAB_E = F.tabla_anticipacion(E["C"], E["code"], E["t_event"], EXP, levels=(2, 3, 4, 5), window_h=120)
TAB_E.to_csv(FIG / "earl2016_huauchinango_anticipacion.csv", index=False)
TAB_E.pivot_table(index="serie", columns="nivel", values="anticipacion_h", sort=False).round(1)

In [ ]:
for tag in ("v020", "p95ss"):
    guardar(F.fig_mapa_deslizamiento(E["C"], E["code"], tiempos_previos(E), tag=tag, mode=MODO_MUNICIPIO),
            f"earl2016_huauchinango_mapas_{tag}")

## 3. Para la discusión

1. **¿ITCHI estaba alto antes del deslizamiento?** Clase de la última posición previa y anticipación del nivel 4 o 5 (tablas 1.2 y 2).
2. **¿Qué medida de lluvia lo dice primero?** Panel b): si R24 o A72 superaron su P95 antes que P6, la escalada por persistencia y suelo saturado es la parte de ITCHI que más importa para deslizamientos; con el máximo en P95 esa escalada actúa desde el P90 (notebook 11, A3).
3. **¿Qué pasa después de la última posición?** El H_P continuo muestra si la lluvia extrema siguió sin ciclón definido. Si siguió, el problema no es del índice sino de la atribución: ITCHI se apaga cuando el ciclón se disipa aunque su humedad siga lloviendo. Una opción para v0.3 es mantener la atribución sobre los remanentes (trayectoria extendida) unas 24–48 h.
4. **Municipio grande.** Atoyac de Álvarez abarca de la costa a la sierra; el máximo municipal puede venir de un píxel costero lejos de La Pintada. Con las coordenadas de la localidad (catálogo de localidades del INEGI) se puede repetir la serie en el píxel del deslizamiento.
5. **Un evento no valida nada.** Estos dos casos ilustran cómo se lee ITCHI ante un impacto concreto. Una evaluación con deslizamientos necesita un inventario (por ejemplo el catálogo de procesos de remoción en masa del CENAPRED) y la misma prueba ordinal que con las declaratorias.